> **Note:** This notebook defines an Airflow DAG for the sandbox environment.
> The code is meant to be read and understood — full execution requires Airflow's scheduler and infrastructure.
> Use the [sandbox Jupyter](https://sandbox.learnmlops.ops4life.com) to run DAGs end-to-end.


# Operations: Drift-Triggered Retraining DAG

This notebook mirrors the **Data Drift & Model Decay** guide at [learnmlops.ops4life.com/guides/data-drift/](https://learnmlops.ops4life.com/guides/data-drift/).

Write an Airflow DAG that runs daily, checks for data drift using PSI, and automatically triggers model retraining when drift exceeds the threshold.

**What we'll cover:**
1. Understand the drift-triggered retraining pattern
2. Write the DAG definition
3. Deploy to `/workspace/dags/`
4. Validate syntax

In [ ]:
import os
import subprocess

os.makedirs('/workspace/dags', exist_ok=True)

result = subprocess.run(['airflow', 'version'], capture_output=True, text=True)
print(result.stdout.strip())

## Step 1: The drift-triggered retraining pattern

```
[Daily schedule]
       |
       v
check_drift (BranchPythonOperator)
       |
       +--- drift detected ---> retrain_model ---> promote_model
       |
       +--- no drift       ---> no_action
```

`BranchPythonOperator` returns the task ID of the branch to follow based on the drift check result. Only that branch's tasks run; the other branch is skipped.

`XCom` passes the PSI value from `check_drift` to `retrain_model` so the run is logged with its trigger condition.

In [ ]:
dag_code = 'from datetime import datetime, timedelta\nimport pandas as pd\nimport numpy as np\nfrom scipy import stats\nfrom airflow import DAG\nfrom airflow.operators.python import PythonOperator, BranchPythonOperator\n\nDATA_DIR = "/workspace/pipeline-data"\nDRIFT_THRESHOLD = 0.20   # PSI threshold for triggering retraining\n\ndefault_args = {\n    "owner": "mlops",\n    "retries": 1,\n    "retry_delay": timedelta(minutes=5),\n}\n\n\ndef _calculate_psi(reference, current, bins=10):\n    ref_hist, bin_edges = np.histogram(reference, bins=bins)\n    cur_hist, _         = np.histogram(current, bins=bin_edges)\n    ref_pct = ref_hist / len(reference) + 1e-10\n    cur_pct = cur_hist / len(current)   + 1e-10\n    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))\n\n\ndef check_drift(**context):\n    import pandas.io.common\n    ref_path = f"{DATA_DIR}/raw_ingested.csv"\n    if not os.path.exists(ref_path):\n        print("Reference data not found — skipping drift check.")\n        return "no_action"\n\n    reference = pd.read_csv(ref_path)\n    np.random.seed(int(datetime.now().timestamp()) % 1000)\n    n = 200\n    current_income = np.random.randint(5000, 25000, n).astype(float)\n    ref_income = reference["MonthlyIncome"].values.astype(float)\n    psi = _calculate_psi(ref_income, current_income)\n    print(f"MonthlyIncome PSI: {psi:.4f}  (threshold: {DRIFT_THRESHOLD})")\n    context["ti"].xcom_push(key="psi", value=psi)\n\n    if psi > DRIFT_THRESHOLD:\n        print("Drift detected — triggering retraining.")\n        return "retrain_model"\n    else:\n        print("No significant drift — skipping retraining.")\n        return "no_action"\n\n\ndef retrain_model(**context):\n    import mlflow\n    import mlflow.sklearn\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.preprocessing import StandardScaler\n    from sklearn.pipeline import Pipeline\n    from sklearn.metrics import roc_auc_score\n\n    mlflow.set_tracking_uri("http://mlflow:5000")\n    mlflow.set_experiment("attrition-drift-retrain")\n\n    train_path = f"{DATA_DIR}/train.csv"\n    if not os.path.exists(train_path):\n        print("train.csv not found — run the pipeline notebooks first.")\n        return\n\n    df = pd.read_csv(train_path)\n    X  = df.drop(columns=["Attrition"])\n    y  = df["Attrition"]\n\n    pipeline = Pipeline([\n        ("scaler", StandardScaler()),\n        ("clf",    LogisticRegression(max_iter=1000, class_weight="balanced")),\n    ])\n\n    psi = context["ti"].xcom_pull(key="psi", task_ids="check_drift")\n    with mlflow.start_run(run_name="drift-triggered-retrain"):\n        mlflow.log_metric("trigger_psi", psi or 0)\n        pipeline.fit(X, y)\n        auc = roc_auc_score(y, pipeline.predict_proba(X)[:, 1])\n        mlflow.log_metric("train_auc", auc)\n        mlflow.sklearn.log_model(pipeline, "model")\n        print(f"Retrained. Train AUC: {auc:.4f}")\n\n\ndef promote_model(**context):\n    import mlflow\n    from mlflow.tracking import MlflowClient\n\n    mlflow.set_tracking_uri("http://mlflow:5000")\n    client = MlflowClient()\n\n    exp = client.get_experiment_by_name("attrition-drift-retrain")\n    if exp is None:\n        print("No retrain experiment found.")\n        return\n\n    runs = client.search_runs(\n        experiment_ids=[exp.experiment_id],\n        order_by=["start_time DESC"],\n        max_results=1,\n    )\n    if not runs:\n        return\n\n    run_id = runs[0].info.run_id\n    mv = mlflow.register_model(f"runs:/{run_id}/model", "attrition-classifier")\n    client.transition_model_version_stage(\n        name="attrition-classifier",\n        version=mv.version,\n        stage="Staging",\n    )\n    print(f"Promoted: attrition-classifier v{mv.version} -> Staging")\n\n\ndef no_action(**context):\n    print("No drift detected. Model remains unchanged.")\n\n\nwith DAG(\n    dag_id="drift_detection_and_retrain",\n    default_args=default_args,\n    schedule_interval="0 8 * * *",\n    start_date=datetime(2025, 1, 1),\n    catchup=False,\n    tags=["mlops", "drift", "retraining"],\n) as dag:\n    import os\n    t_check   = BranchPythonOperator(task_id="check_drift",   python_callable=check_drift)\n    t_retrain = PythonOperator(task_id="retrain_model",       python_callable=retrain_model)\n    t_promote = PythonOperator(task_id="promote_model",       python_callable=promote_model)\n    t_no_act  = PythonOperator(task_id="no_action",           python_callable=no_action)\n\n    t_check >> [t_retrain, t_no_act]\n    t_retrain >> t_promote'

print('DAG code preview (first 15 lines):')
print('\n'.join(dag_code.split('\n')[:15]))
print('...')

## Step 2: Key design decisions

- **`BranchPythonOperator`** — returns the task ID of the next branch; Airflow skips all tasks not in that branch
- **XCom** — PSI value is pushed in `check_drift` and pulled in `retrain_model` so the retrain task can log what triggered it
- **MLflow tracking URI = `http://mlflow:5000`** — uses the sandbox MLflow service, not a Kubernetes endpoint
- **No S3** — reference data and training data are read from `/workspace/pipeline-data/` on the shared Docker volume

## Step 3: Deploy to the DAGs folder

In [ ]:
dag_path = '/workspace/dags/drift_retrain_dag.py'
with open(dag_path, 'w') as f:
    f.write(dag_code)
print(f'DAG deployed: {dag_path}')

## Step 4: Validate syntax

In [ ]:
result = subprocess.run(['python', dag_path], capture_output=True, text=True)
if result.returncode == 0:
    print('Syntax OK — no errors')
else:
    print('Syntax error:')
    print(result.stderr)

Open [airflow.learnmlops.ops4life.com](https://airflow.learnmlops.ops4life.com) and look for **drift_detection_and_retrain** in the DAG list (appears within 30 seconds). Trigger it manually to see the branch execute: either `retrain_model → promote_model` or `no_action` depending on the current PSI value.

## Next Steps

- Return to the beginning: `01-foundation/ml-basics/train_iris.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/data-drift/](https://learnmlops.ops4life.com/guides/data-drift/)